# Bivariate Bicycle Code — [[144, 12, 12]]

This notebook walks through the construction and performance of the **[[144, 12, 12]] bivariate bicycle (BB) code** from Bravyi et al. (arXiv:2308.07915).

BB codes are CSS quantum LDPC codes defined by two polynomials over a finite ring. The [[144,12,12]] code encodes **12 logical qubits** with distance 12 into just 144 physical qubits — a much better encoding rate than the surface code.

### The math
Define two trinomials over $\mathbb{F}_2[x,y]\,/\,(x^{\ell}-1,\, y^m-1)$:

$$A(x,y) = x^3 + y + y^2, \qquad B(x,y) = y^3 + x + x^2$$

For $\ell=12,\ m=6$ this gives the [[144,12,12]] code. The parity-check matrices are:

$$H_X = [A\mid B], \qquad H_Z = [B^\top \mid A^\top]$$

Each is a $72\times144$ matrix over $\mathbb{F}_2$. Both stabiliser types have **weight 6** (one term per monomial in $A$ and $B$).

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import matplotlib.pyplot as plt
import stim

from bb_code_sim import (
    BB_144_12_12,
    build_parity_checks,
    find_logical_ops,
    BBCodeSimulator,
    RelayBPDecoder,
)
from surface_code_sim import ErrorModel

params = BB_144_12_12
print(f"Code parameters: l={params.l}, m={params.m}, distance={params.distance}")
print(f"Physical qubits n = 2·l·m = {2 * params.l * params.m}")

## 1. Parity-Check Matrices

We construct $H_X$ and $H_Z$ by turning each monomial $x^a y^b$ into a circulant shift on the $\ell\times m$ torus. Each check (row) has exactly 6 non-zero entries.

In [ ]:
H_X, H_Z = build_parity_checks(params)

n_checks, n_data = H_X.shape
print(f"H_X shape: {H_X.shape}  ({n_checks} X-checks, {n_data} data qubits)")
print(f"H_Z shape: {H_Z.shape}")
print(f"\nCSS condition H_X · H_Z^T = 0 (mod 2): {np.all(H_X @ H_Z.T % 2 == 0)}")

In [ ]:
# Every row of H_X and H_Z should have weight exactly 6
row_weights_X = H_X.sum(axis=1)
row_weights_Z = H_Z.sum(axis=1)
col_weights_X = H_X.sum(axis=0)
col_weights_Z = H_Z.sum(axis=0)

print(f"X-check weights: min={row_weights_X.min()}, max={row_weights_X.max()} (all should be 6)")
print(f"Z-check weights: min={row_weights_Z.min()}, max={row_weights_Z.max()} (all should be 6)")
print(f"\nData qubit participation in X-checks: min={col_weights_X.min()}, max={col_weights_X.max()} (all should be 3)")
print(f"Data qubit participation in Z-checks: min={col_weights_Z.min()}, max={col_weights_Z.max()} (all should be 3)")

### Visualising $H_X$

We can visualise the sparsity structure — note how the non-zeros form circulant diagonal bands (the toric periodicity).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, H, label in zip(axes, [H_X, H_Z], ["$H_X$", "$H_Z$"]):
    ax.spy(H, markersize=1.5, color='steelblue')
    ax.set_title(f"{label}  ({H.shape[0]}×{H.shape[1]}, weight-6 rows)", fontsize=12)
    ax.set_xlabel("Data qubit index")
    ax.set_ylabel("Check index")

plt.tight_layout()
plt.show()

## 2. Logical Operators

For a CSS code with parity checks $H_X, H_Z$:
- **Logical Z operators** span $\ker(H_X) \,/\, \operatorname{rowspace}(H_Z)$
- **Logical X operators** span $\ker(H_Z) \,/\, \operatorname{rowspace}(H_X)$

Both quotient spaces should have dimension $k = n - \operatorname{rank}(H_X) - \operatorname{rank}(H_Z)$.

In [ ]:
log_Z, log_X = find_logical_ops(H_X, H_Z)
k = len(log_Z)

print(f"Logical qubits encoded: k = {k}")
print(f"Logical Z operators shape: {log_Z.shape}")
print(f"Logical X operators shape: {log_X.shape}")
print()

# Verify the logical operators commute with all stabilisers
assert np.all(H_X @ log_Z.T % 2 == 0), "Logical Z not in kernel of H_X!"
assert np.all(H_Z @ log_X.T % 2 == 0), "Logical X not in kernel of H_Z!"
print("Logical Z ops commute with all X-checks: ✓")
print("Logical X ops commute with all Z-checks: ✓")

# Anti-commutation table: log_Z[i] · log_X[j] should be delta_ij
anticomm = log_Z @ log_X.T % 2
print(f"\nAnti-commutation table (should be identity):")
print(anticomm)

## 3. Stim Circuit

We build a **Z-memory experiment**: initialise all data qubits in $|0\rangle$, run $r$ rounds of syndrome extraction, then measure all data qubits in the Z basis. Logical Z errors are tracked.

The qubit layout is:
- `[0, 144)` — 144 data qubits
- `[144, 216)` — 72 X-ancilla
- `[216, 288)` — 72 Z-ancilla

Noise model: circuit-level depolarising. Each CNOT gets `DEPOLARIZE2(p)`, single-qubit gates get `DEPOLARIZE1(p)`, measurements/resets get `X_ERROR(p_meas)`.

In [ ]:
sim = BBCodeSimulator(params)
em_example = ErrorModel.symmetric(0.001)
circ = sim.build_circuit(em_example, rounds=12)

print(f"Total qubits : {circ.num_qubits}")
print(f"Detectors    : {circ.num_detectors}")
print(f"Observables  : {circ.num_observables}  (= k = {k})")
print()
print("First 30 instructions:")
for instr in list(circ)[:30]:
    print(' ', instr)

### Zero-noise sanity check

With no noise, the decoder should predict zero logical errors on every shot.

In [ ]:
result_zero = sim.run(
    ErrorModel.symmetric(0.0),
    rounds=12,
    shots=50,
    seed=0,
)
print(result_zero)
assert result_zero.logical_error_rate == 0.0, "Non-zero LER in noiseless circuit!"
print("Sanity check passed ✓")

## 4. Logical Error Rate vs Syndrome Rounds

For a memory experiment, the logical error rate should grow with the number of syndrome rounds. At low noise it should grow roughly linearly (one "spacetime slab" per round).

> **Decoder:** [Relay-BP](https://github.com/trmue/relay) (Rust, batched) — the default decoder for BB codes here. ~22× faster than BP-OSD at realistic error rates.

In [ ]:
import time
from tqdm import tqdm

round_values = [1, 3, 6, 12, 18]
em_fixed = ErrorModel.symmetric(0.001)

rounds_results = []
t_total = 0
for r in tqdm(round_values, desc="sweep_rounds", unit="round"):
    t0 = time.perf_counter()
    result = sim.run(em_fixed, rounds=r, shots=200, seed=1)
    elapsed = time.perf_counter() - t0
    t_total += elapsed
    rounds_results.append(result)
    print(f"  rounds={r:2d}  {elapsed:6.1f}s  {result}")

print(f"\nTotal: {t_total:.1f}s  ({t_total/len(round_values):.1f}s/point)")

In [ ]:
lers  = [r.logical_error_rate    for r in rounds_results]
errs  = [r.logical_error_rate_se for r in rounds_results]

plt.figure(figsize=(6, 4))
plt.errorbar(round_values, lers, yerr=errs, fmt='o-', capsize=4,
             color='steelblue', label='[[144,12,12]] BB code')
plt.xlabel("Syndrome rounds")
plt.ylabel("Logical error rate")
plt.title(f"LER vs rounds  (p = {em_fixed.p_phys:.3g})")
plt.ylim(bottom=0)
plt.legend()
plt.tight_layout()
plt.show()

## 5. Logical Error Rate vs Physical Error Rate

Below we sweep the physical error rate $p$ at a fixed number of syndrome rounds equal to the code distance ($r = 12$). We expect the LER to drop steeply for $p$ below the pseudo-threshold, reflecting the code's error-correcting power.

In [ ]:
import time
from tqdm import tqdm

# Shots tuned so each point runs in ~60s:
# lower p → BP converges fast → more shots for statistical power
# higher p → BP struggles → fewer shots to keep runtime reasonable
p_shots = {
    0.002: 500,
    0.003: 300,
    0.004: 150,
    0.005: 100,
    0.007:  50,
    0.010:  30,
    0.015:  20,
}

p_sweep_results = []
t_total = 0
for p, shots in tqdm(p_shots.items(), desc="sweep_p", unit="point"):
    em = ErrorModel.symmetric(p)
    t0 = time.perf_counter()
    result = sim.run(em, rounds=params.distance, shots=shots, seed=2)
    elapsed = time.perf_counter() - t0
    t_total += elapsed
    p_sweep_results.append(result)
    print(f"  p={p:.3f}  shots={shots}  {elapsed:.1f}s  {result}")

print(f"\nTotal: {t_total:.1f}s")

In [ ]:
lers = [r.logical_error_rate    for r in p_sweep_results]
errs = [r.logical_error_rate_se for r in p_sweep_results]

plt.figure(figsize=(6, 4))
plt.errorbar(p_values, lers, yerr=errs, fmt='o-', capsize=4, color='steelblue',
             label=f'[[144,12,12]]  d={params.distance} rounds')
plt.axhline(0.5, color='grey', linestyle='--', linewidth=0.8, label='50% (random guess)')
plt.xscale('log')
plt.yscale('log')
plt.xlabel("Physical error rate p")
plt.ylabel("Logical error rate")
plt.title("[[144,12,12]] BB code — LER vs p")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Comparing [[72,12,6]] and [[144,12,12]]

Both codes use the same polynomials $A$ and $B$ — the [[144,12,12]] code is essentially a "double cover" of [[72,12,6]] with $\ell$ doubled from 6 to 12. Both encode 12 logical qubits but at different distances and physical qubit costs.

Running both at matched rounds/distance lets us see the effect of the larger code.

In [ ]:
from bb_code_sim import BB_72_12_6

p_values_cmp = [0.001, 0.002, 0.004, 0.006, 0.008, 0.01]

results_72  = BBCodeSimulator(BB_72_12_6).sweep_p(
    p_values_cmp, rounds=BB_72_12_6.distance, shots=200, seed=3)
results_144 = BBCodeSimulator(BB_144_12_12).sweep_p(
    p_values_cmp, rounds=BB_144_12_12.distance, shots=200, seed=3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

for results, label, color in [
    (results_72,  '[[72,12,6]]   (d=6)',  'tomato'),
    (results_144, '[[144,12,12]] (d=12)', 'steelblue'),
]:
    lers = [r.logical_error_rate    for r in results]
    errs = [r.logical_error_rate_se for r in results]
    ax.errorbar(p_values_cmp, lers, yerr=errs, fmt='o-', capsize=4,
                label=label, color=color)

ax.axhline(0.5, color='grey', linestyle='--', linewidth=0.8, label='50%')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel("Physical error rate p")
ax.set_ylabel("Logical error rate")
ax.set_title("Bivariate Bicycle codes — LER vs p")
ax.legend()
plt.tight_layout()
plt.show()

## Summary

| Property | [[72, 12, 6]] | [[144, 12, 12]] |
|----------|--------------|----------------|
| Physical qubits | 72 | 144 |
| Logical qubits | 12 | 12 |
| Distance | 6 | 12 |
| Encoding rate | 1/6 | 1/12 |
| Check weight | 6 | 6 |
| Qubit degree | 3 per check type | 3 per check type |

For comparison, a surface code with the same distance-12 protection needs $d^2 = 144$ physical qubits for **1 logical qubit** — the [[144,12,12]] code achieves the same physical qubit count while encoding **12 logical qubits**.

### Decoder note
We use **Relay-BP** ([trmue/relay](https://github.com/trmue/relay)), a Rust-native decoder that chains multiple disordered-memory BP runs to escape local minima. It is the default decoder (`RelayBPDecoder`) for all BB code simulations in this notebook.